# Notebook 3: BERTopic — Unsupervised Topic Discovery

This notebook explains **how BERTopic discovers topics without any labels**, why it works better than older approaches, and what the output means.

Covers ADRs 008–011.

---

## Why Unsupervised Topic Discovery?

We have 142,000 news articles and **no labels**. We don't know in advance what topics exist. Instead of manually defining categories and labeling thousands of articles (expensive and subjective), we let the model *discover* the topics automatically.

This is the key insight of the **hybrid approach**:
```
Step 1: BERTopic discovers topics automatically  (unsupervised — no labels needed)
                          │
                          ▼
Step 2: Use those discovered topics as labels   (now we have a labeled dataset)
                          │
                          ▼
Step 3: Train a fast classifier (Logistic Regression) to predict topics on new articles
```

**Why not just use BERTopic for everything?**  
BERTopic is slow — it must embed every article. A trained Logistic Regression predicts in milliseconds. The hybrid approach gets the best of both: semantically meaningful topics from BERTopic, fast inference from the classifier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')

# Load topic info and labeled articles
topic_info = pd.read_csv('../data/processed/topic_info.csv')
df = pd.read_parquet('../data/processed/articles_with_topics.parquet')

print(f'Topics discovered: {len(topic_info[topic_info["Topic"] != -1])}')
print(f'Articles with valid topic labels: {len(df):,}')

---

## ADR-008: Why BERTopic over LDA or NMF?

### The older approach: LDA (Latent Dirichlet Allocation)

LDA was the standard topic model for a decade. It works by treating each document as a mixture of topics, and each topic as a distribution over words — all based on word counts, not meaning.

**The problem:** LDA doesn't know that "car" and "automobile" mean the same thing. It treats them as unrelated tokens.

### The BERTopic approach

BERTopic uses **sentence embeddings** — vector representations that capture *semantic meaning*. Articles about similar things land near each other in embedding space, regardless of whether they use the exact same words.

```
BERTopic pipeline:

Articles  →  [Sentence Transformer]  →  dense vectors (384 dims)
                                               │
                                        [UMAP: 384→5 dims]
                                               │
                                        [HDBSCAN clustering]
                                               │
                                        [c-TF-IDF: label each cluster]
                                               │
                                          Topic keywords
```

In [ ]:
# ── What are sentence embeddings? ───────────────────────────────────────────
# Think of embeddings as coordinates in a high-dimensional space.
# Semantically similar sentences have coordinates close together.

# Let's visualize this with a tiny toy example
# (We'll use random 2D coords to illustrate — real embeddings are 384-dim)

toy_articles = {
    'Fed raises rates': (0.1, 0.2),
    'Interest rate hike': (0.15, 0.25),
    'Inflation concerns': (0.2, 0.15),
    'Trump signs bill': (0.8, 0.7),
    'Senate vote today': (0.75, 0.8),
    'Congress passes law': (0.85, 0.75),
    'NBA finals recap': (0.1, 0.9),
    'Lakers win game': (0.15, 0.85),
    'Football season starts': (0.2, 0.95),
}

colors = ['steelblue']*3 + ['coral']*3 + ['green']*3
labels = ['Economics']*3 + ['Politics']*3 + ['Sports']*3

fig, ax = plt.subplots(figsize=(8, 6))
for (title, (x, y)), color, label in zip(toy_articles.items(), colors, labels):
    ax.scatter(x, y, color=color, s=200, zorder=5)
    ax.annotate(title, (x, y), textcoords='offset points', xytext=(8, 4), fontsize=9)

patches = [mpatches.Patch(color=c, label=l) for c, l in [('steelblue','Economics'),('coral','Politics'),('green','Sports')]]
ax.legend(handles=patches)
ax.set_title('Toy Embedding Space — Similar Articles Cluster Together', fontsize=13)
ax.set_xlabel('Embedding dimension 1')
ax.set_ylabel('Embedding dimension 2')
ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

print('HDBSCAN finds the dense clusters automatically — no labels needed.')

In [ ]:
# ── What topics did BERTopic discover in our real dataset? ──────────────────
real_topics = topic_info[topic_info['Topic'] != -1][['Topic', 'Count', 'Name']].copy()
real_topics = real_topics.sort_values('Count', ascending=False).reset_index(drop=True)

# Clean up the name for display
real_topics['Keywords'] = real_topics['Name'].str.split('_').str[1:].str.join(', ')

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(real_topics['Keywords'], real_topics['Count'], color=sns.color_palette('muted', len(real_topics)))
ax.set_title('Articles per Discovered Topic', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Articles')
ax.invert_yaxis()
for bar, count in zip(bars, real_topics['Count']):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, f'{count:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

> **Interview Talking Point (ADR-008):**  
> *"I chose BERTopic over LDA because it uses sentence embeddings rather than word counts. This means semantically similar articles cluster together even if they use different vocabulary. LDA would separate 'automobile recall' and 'car safety issue' into different topics — BERTopic groups them correctly."*

---

## How Each BERTopic Step Works

### Step 1: Sentence Embeddings (ADR-010)

In [ ]:
# The embedding model converts each article to a 384-dimensional vector
# We used all-MiniLM-L6-v2 — fast, high quality, industry standard for BERTopic

# Each article becomes a point in 384-dimensional space
# Similar articles are geometrically close to each other

print('Embedding model: all-MiniLM-L6-v2')
print('  Output dimensions: 384')
print('  Speed on Apple MPS: ~7-8 batches/sec')
print()
print('Why this model?')
print('  • 2x faster than 768-dim models (all-mpnet-base-v2)')
print('  • Small quality reduction — acceptable for topic discovery')
print('  • Default for BERTopic — community-validated')
print('  • Automatically uses MPS on Apple Silicon (no extra config)')

### Step 2: UMAP — Dimensionality Reduction

UMAP compresses 384 dimensions down to 5. Why?

- **Clustering in 384 dimensions is hard** — in very high-dimensional spaces, all points tend to be roughly equidistant from each other (the "curse of dimensionality"). HDBSCAN can't find meaningful clusters.
- **UMAP preserves local structure** — articles that were close in 384D stay close in 5D. Clusters survive the compression.
- **Why 5 and not 2?** 2D is for visualization. 5D preserves more structure for clustering without the curse of dimensionality.

In [ ]:
# Visualize: the curse of dimensionality
# In high dimensions, random points are all roughly the same distance apart

dims = [2, 5, 10, 50, 100, 384]
results = []
np.random.seed(42)
for d in dims:
    points = np.random.randn(1000, d)
    dists = np.linalg.norm(points[:100][:, None] - points[None, :100], axis=-1)
    np.fill_diagonal(dists, np.nan)
    mean_d = np.nanmean(dists)
    std_d  = np.nanstd(dists)
    results.append({'dims': d, 'mean_dist': mean_d, 'std_dist': std_d, 'cv': std_d/mean_d})

res_df = pd.DataFrame(results)
print('Coefficient of variation (std/mean) of pairwise distances:')
print('Lower = harder to find clusters (all points look equally far apart)')
print()
for _, row in res_df.iterrows():
    bar = '█' * int(row['cv'] * 100)
    print(f'  {int(row["dims"]):>4} dims → CV={row["cv"]:.3f}  {bar}')

print()
print('UMAP reduces 384→5 dims so HDBSCAN can find meaningful clusters.')

### Step 3: HDBSCAN — Clustering

In [ ]:
# HDBSCAN finds dense regions in the UMAP-reduced space
# Key advantage over k-means: it doesn't require you to specify the number of clusters
# It finds however many natural clusters exist in the data

print('HDBSCAN vs k-means:')
print()
comparison = pd.DataFrame({
    'Feature': ['Specify # clusters?', 'Handles noise?', 'Cluster shapes', 'Outlier label'],
    'k-means': ['Yes (must guess)', 'No (every point assigned)', 'Spherical only', 'None — forces assignment'],
    'HDBSCAN': ['No (discovers automatically)', 'Yes (topic -1)', 'Any shape', 'Topic -1 = outlier'],
})
print(comparison.to_string(index=False))

### Step 4: c-TF-IDF — Labeling Each Cluster

In [ ]:
# After clustering, BERTopic labels each cluster using c-TF-IDF
# (class-based TF-IDF — treat all articles in a cluster as one big document)
# The words that appear often in THIS cluster but rarely in OTHER clusters
# become the topic keywords

print('Discovered topic keywords (each topic = a cluster of similar articles):')
print()
for _, row in real_topics.head(19).iterrows():
    print(f'  Topic {int(row["Topic"]):>2}  ({row["Count"]:>5,} articles)  {row["Keywords"]}')

---

## ADR-009: Why Fit on a Sample?

Embedding 141k articles end-to-end takes ~30-60 minutes. The production pattern is: **fit on a representative sample, transform the rest.**

In [ ]:
# Show the time breakdown
print('Time breakdown for our run:')
print()
breakdown = pd.DataFrame({
    'Step': [
        'Embed 20k sample (MPS)',
        'UMAP 384→5 dims',
        'HDBSCAN clustering',
        'Transform remaining 121k articles',
    ],
    'Approx time': ['~3-4 min', '~1-2 min', '~30 sec', '~8-10 min'],
    'Notes': [
        'MPS-accelerated on Apple Silicon',
        'Only runs on the 20k sample',
        'Only runs on the 20k sample',
        'Only embedding — no UMAP/HDBSCAN re-fit',
    ]
})
print(breakdown.to_string(index=False))

print()
print('If we ran fit_transform on all 141k:')
print('  Embedding alone would take ~30-50 min')
print('  UMAP on 141k × 384 would require ~10-15GB RAM')

In [ ]:
# Why stratified sampling?
# We want the 20k sample to represent all 15 publications proportionally
# Otherwise Breitbart (23k articles) would dominate the topic discovery

full_pub_dist = pd.read_parquet('../data/processed/articles_clean.parquet')['publication'].value_counts(normalize=True)
print('Publication distribution in full dataset (% of articles):')
print((full_pub_dist * 100).round(1).to_string())

> **Interview Talking Point (ADR-009):**  
> *"Running BERTopic on all 141k articles would require 30+ minutes of embedding time and ~15GB of RAM for UMAP. Instead, I fit BERTopic on a stratified 20k-article sample — stratified by publication so all outlets are proportionally represented — then used `.transform()` to assign topics to the remaining articles. `.transform()` only does embedding and nearest-centroid lookup, skipping the expensive UMAP/HDBSCAN re-fit."*

---

## ADR-011: Handling Outliers (Topic -1)

In [ ]:
# HDBSCAN assigns topic -1 to articles that don't fit any cluster
# This is a feature, not a bug — it means HDBSCAN is being honest
# about articles it can't confidently categorize

total_articles = 141112
outliers = 53720
labeled = total_articles - outliers

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Valid topic label', 'Outlier (topic -1)'],
       [labeled, outliers],
       color=['steelblue', 'salmon'])
ax.set_title('Article Label Status After BERTopic', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Articles')
for i, v in enumerate([labeled, outliers]):
    ax.text(i, v + 500, f'{v:,}\n({100*v/total_articles:.0f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Why drop outliers instead of assigning them to the nearest topic?

print('The tradeoff: noisy labels vs smaller dataset')
print()
print('Option A — Keep outliers, assign to nearest topic:')
print(f'  Training set: {total_articles:,} articles')
print(f'  Problem: 53,720 articles have wrong/forced labels → degrades classifier accuracy')
print()
print('Option B — Drop outliers (our choice):')
print(f'  Training set: {labeled:,} articles')
print(f'  All labels are high-confidence (HDBSCAN assigned them to a real cluster)')
print(f'  Clean labels → better classifier generalization')
print()
print('General principle: small + clean beats large + noisy for supervised learning')

> **Interview Talking Point (ADR-011):**  
> *"HDBSCAN assigned ~38% of articles to topic -1, meaning it couldn't confidently place them in any cluster. I dropped those rather than force-assigning them to the nearest topic. Noisy labels degrade classifier accuracy more than a smaller training set hurts it — 87k clean examples trains a better model than 141k with 38% mislabeled."*

---

## What Gets Passed to the Next Stage?

The output of `topic_model.py` is `articles_with_topics.parquet` — 87,392 articles, each with a discovered topic label. This becomes the training data for the supervised classifier.

In [ ]:
print('Labeled article sample:')
print(df[['title', 'publication', 'topic_id', 'topic_label']].sample(8, random_state=1).to_string(index=False))

print(f'\nClass distribution (articles per topic):')
topic_counts = df['topic_label'].value_counts()
print(topic_counts.to_string())
print(f'\nMin: {topic_counts.min():,}  |  Max: {topic_counts.max():,}  |  Ratio: {topic_counts.max()/topic_counts.min():.1f}x')

---

## Summary: BERTopic End to End

```
141k articles
      │
      ├── 20k stratified sample
      │         │
      │   [sentence-transformers: all-MiniLM-L6-v2]
      │         │  384-dim vectors (MPS-accelerated)
      │         ▼
      │   [UMAP: 384 → 5 dims]
      │         │  preserves local cluster structure
      │         ▼
      │   [HDBSCAN: min_cluster_size=50]
      │         │  finds 19 natural clusters + outliers
      │         ▼
      │   [c-TF-IDF: label each cluster with keywords]
      │
      └── remaining 121k articles
                │
          [.transform(): embed + nearest-centroid assign]
                │
                ▼
      87,392 articles with clean topic labels
      (53,720 outliers dropped)
                │
                ▼
      → train.py: Logistic Regression classifier
```